# Notebook 01 — LINE OA & Webhook

## LINE OA คืออะไร?

**LINE Official Account (LINE OA)** คือบัญชีธุรกิจบน LINE ที่ให้บริษัทหรือผู้พัฒนาสร้าง chatbot เพื่อโต้ตอบกับผู้ใช้งาน

### วิธีที่ LINE OA ทำงาน

```
1. ผู้ใช้ส่งข้อความ ──► LINE Platform (Server)
2. LINE Platform ──► Webhook URL ของเรา (HTTP POST)
3. Server เราประมวลผล ──► ส่งคำตอบกลับผ่าน Reply API
4. LINE Platform ──► ส่งข้อความถึงผู้ใช้
```

### Key Concepts

| Concept | คำอธิบาย |
|---------|----------|
| **Webhook URL** | URL ที่ LINE จะ POST event มาให้ |
| **Channel Access Token** | Token สำหรับเรียก LINE API |
| **Channel Secret** | ใช้ verify signature ว่า request มาจาก LINE จริง |
| **Reply Token** | Token ชั่วคราวสำหรับตอบกลับแต่ละ event |
| **User ID** | ID เฉพาะของผู้ใช้แต่ละคน (เช่น `U1234abcd...`) |
| **Postback** | Event ที่เกิดจากการกดปุ่มพิเศษ (Rich Menu, Flex Message) |


## รูปแบบของ LINE Webhook Event

เมื่อผู้ใช้ส่งข้อความ LINE จะส่ง JSON นี้มาที่ webhook URL เรา:

In [ ]:
# ตัวอย่าง LINE Webhook Payload (Text Message)
example_text_event = {
    "destination": "Uxxxxxxxxxxxxx",
    "events": [
        {
            "type": "message",
            "message": {
                "type": "text",
                "id": "111111111111",
                "text": "สวัสดีครับ"  # ข้อความจากผู้ใช้
            },
            "webhookEventId": "01ABCDEF...",
            "deliveryContext": {"isRedelivery": False},
            "timestamp": 1713000000000,
            "source": {
                "type": "user",
                "userId": "U1234567890abcdef"  # LINE User ID
            },
            "replyToken": "nHuyWiB7yP5Zw52FIkcQobQuGDXCTA",  # ใช้ตอบกลับ
            "mode": "active"
        }
    ]
}

# ตัวอย่าง Postback Event (กดปุ่มใน Rich Menu หรือ Flex)
example_postback_event = {
    "destination": "Uxxxxxxxxxxxxx",
    "events": [
        {
            "type": "postback",
            "postback": {
                "data": "action=concierge_menu"  # data ที่กำหนดใน button
            },
            "source": {
                "type": "user",
                "userId": "U1234567890abcdef"
            },
            "replyToken": "nHuyWiB7yP5Zw52FIkcQobQuGDXCTA",
            "timestamp": 1713000000000,
            "mode": "active"
        }
    ]
}

print("Text Event type:", example_text_event['events'][0]['type'])
print("Text:", example_text_event['events'][0]['message']['text'])
print("User ID:", example_text_event['events'][0]['source']['userId'])
print()
print("Postback data:", example_postback_event['events'][0]['postback']['data'])

## Extract LINE Event — `src/integrations/line/utils.py`

โปรเจคนี้มี function `extract_line_event()` ที่แปลง raw JSON จาก LINE เป็น dict ที่ใช้งานง่ายขึ้น:

In [ ]:
# จำลองการทำงานของ extract_line_event()
# ดูโค้ดจริงได้ที่: src/integrations/line/utils.py

def extract_line_event(raw_body: dict) -> dict | None:
    """
    แปลง raw LINE webhook body เป็น dict ที่ normalize แล้ว
    รองรับ: message (text), postback
    """
    events = raw_body.get("events", [])
    if not events:
        return None
    
    event = events[0]  # ใช้ event แรก
    event_type = event.get("type")  # "message" หรือ "postback"
    
    result = {
        "event_type": event_type,
        "reply_token": event.get("replyToken"),
        "user_id": event.get("source", {}).get("userId"),
        "message_text": None,
        "postback_data": None,
    }
    
    if event_type == "message":
        msg = event.get("message", {})
        if msg.get("type") == "text":
            result["message_text"] = msg.get("text")
    
    elif event_type == "postback":
        result["postback_data"] = event.get("postback", {}).get("data")
    
    return result


# ทดสอบกับ text event
result = extract_line_event(example_text_event)
print("Text event result:")
for k, v in result.items():
    print(f"  {k}: {v}")

print()

# ทดสอบกับ postback event
result2 = extract_line_event(example_postback_event)
print("Postback event result:")
for k, v in result2.items():
    print(f"  {k}: {v}")

## Reply to LINE — `src/integrations/line/messaging.py`

หลังจากประมวลผลแล้ว เราต้องส่งคำตอบกลับไปหาผู้ใช้ผ่าน LINE Reply API

In [ ]:
# จำลองการทำงานของ reply_to_line()
# โค้ดจริง: src/integrations/line/messaging.py

import httpx

LINE_REPLY_API = "https://api.line.me/v2/bot/message/reply"

async def reply_to_line(
    reply_token: str,
    text: str | None = None,
    line_messages: list | None = None,
    quick_replies: list | None = None,
    channel_access_token: str = ""
):
    """
    ส่งข้อความตอบกลับไปหาผู้ใช้บน LINE
    
    - text: ข้อความธรรมดา
    - line_messages: Flex Message หรือ complex messages
    - quick_replies: ปุ่มตอบด่วน (Quick Reply)
    """
    messages = []
    
    # สร้าง text message
    if text:
        msg = {"type": "text", "text": text}
        if quick_replies:
            msg["quickReply"] = {"items": quick_replies}
        messages.append(msg)
    
    # เพิ่ม Flex messages
    if line_messages:
        messages.extend(line_messages[:5])  # LINE จำกัดสูงสุด 5 messages ต่อ reply
    
    payload = {
        "replyToken": reply_token,
        "messages": messages
    }
    
    headers = {
        "Authorization": f"Bearer {channel_access_token}",
        "Content-Type": "application/json"
    }
    
    # ส่ง HTTP POST ไปที่ LINE API
    # async with httpx.AsyncClient() as client:
    #     response = await client.post(LINE_REPLY_API, json=payload, headers=headers)
    
    print("Payload ที่จะส่งไป LINE API:")
    import json
    print(json.dumps(payload, ensure_ascii=False, indent=2))


# ตัวอย่างการใช้งาน
import asyncio

asyncio.run(reply_to_line(
    reply_token="nHuyWiB7yP5Zw52FIkcQobQuGDXCTA",
    text="สวัสดีครับ! มีอะไรให้ช่วยไหม?",
    quick_replies=[
        {
            "type": "action",
            "action": {
                "type": "message",
                "label": "ข้อมูลโรงพยาบาล",
                "text": "ข้อมูลโรงพยาบาล"
            }
        }
    ]
))

## Loading Animation

LINE รองรับ **Loading Animation** (จุด 3 จุด) ขณะที่ bot กำลังประมวลผล
ช่วยให้ผู้ใช้รู้ว่า bot กำลังคิดอยู่ ไม่ใช่แฮงก์

In [ ]:
# จำลอง show_loading_animation()
# โค้ดจริง: src/integrations/line/messaging.py

LINE_LOADING_API = "https://api.line.me/v2/bot/chat/loading/start"

async def show_loading_animation(user_id: str, channel_access_token: str):
    """
    แสดงจุด 3 จุดบน LINE chat ขณะที่ bot กำลังประมวลผล
    ต้องส่งก่อนที่จะเริ่ม AI processing
    """
    payload = {
        "chatId": user_id,
        "loadingSeconds": 20  # แสดงนาน 20 วินาที (หรือจนกว่าจะ reply)
    }
    headers = {
        "Authorization": f"Bearer {channel_access_token}",
        "Content-Type": "application/json"
    }
    # await client.post(LINE_LOADING_API, json=payload, headers=headers)
    print(f"✓ Showing loading animation for user: {user_id}")


# Flow ที่ถูกต้อง:
print("Flow การแสดง Loading Animation:")
print("1. รับ webhook event จาก LINE")
print("2. ทันที! เรียก show_loading_animation() ← ผู้ใช้เห็นจุด 3 จุด")
print("3. เริ่ม AI processing (อาจใช้เวลาหลายวินาที)")
print("4. เมื่อได้คำตอบ เรียก reply_to_line() ← จุด 3 จุดหายไป ข้อความปรากฏ")

## Postback vs Text Message

โปรเจคนี้รองรับ 2 ประเภท event หลัก:

In [ ]:
# สรุปความแตกต่าง

comparison = {
    "Text Message": {
        "trigger": "ผู้ใช้พิมพ์ข้อความเอง",
        "event_type": "message",
        "data": "message.text",
        "ตัวอย่าง": "'อยากรู้เรื่องค่าที่จอดรถ'",
        "การประมวลผล": "ส่งเข้า LLM เพื่อวิเคราะห์ intent"
    },
    "Postback": {
        "trigger": "กดปุ่มใน Rich Menu หรือ Flex Message",
        "event_type": "postback",
        "data": "postback.data (เช่น action=concierge_menu)",
        "ตัวอย่าง": "'action=hospital_agent&hospital_id=H001'",
        "การประมวลผล": "Fast-track ไปยัง module โดยตรง ไม่ต้องผ่าน LLM"
    }
}

for event_type, details in comparison.items():
    print(f"\n{'='*40}")
    print(f"Type: {event_type}")
    for k, v in details.items():
        print(f"  {k}: {v}")

print("\n\nPostback ใน โปรเจคนี้ใช้สำหรับ:")
postback_actions = [
    "concierge_menu       → เปิดเมนูหลัก Concierge",
    "hospital_agent       → เข้าสู่ Hospital Info module",
    "nurse_agent          → เข้าสู่ Nurse module",
    "concierge_topic      → เลือก topic ใน Concierge",
    "ai_mode              → ออกจาก Handoff mode กลับมา AI mode"
]
for action in postback_actions:
    print(f"  • {action}")

## Dual Account Setup

โปรเจคนี้รองรับ **2 LINE accounts** พร้อมกัน: DEV และ OFFICIAL

In [ ]:
# การตั้งค่า Dual Account

print("Webhook Endpoints:")
print("  POST /webhook-dev      → DEV account")
print("  POST /webhook-official → OFFICIAL account")
print()

print("ENV Variables:")
print("  LINE_CHANNEL_ACCESS_TOKEN_DEV      = token สำหรับ dev bot")
print("  LINE_CHANNEL_ACCESS_TOKEN_OFFICIAL = token สำหรับ official bot")
print()

print("ทำไมถึงแยก?")
print("  - DEV: ทดสอบ feature ใหม่ ไม่กระทบ user จริง")
print("  - OFFICIAL: production ใช้งานจริง")
print()

print("Token routing ใน webhook_router.py:")
code = '''
@router.post("/webhook-dev")
async def webhook_dev(request: Request):
    body = await request.json()
    for event in body.get("events", []):
        # Attach dev token ไปกับ event
        event["__line_channel_access_token"] = DEV_TOKEN
        event["__line_account"] = "dev"
        asyncio.create_task(_process_one_event(event))
    return {"message": "OK"}

@router.post("/webhook-official")
async def webhook_official(request: Request):
    body = await request.json()
    for event in body.get("events", []):
        # Attach official token ไปกับ event
        event["__line_channel_access_token"] = OFFICIAL_TOKEN
        event["__line_account"] = "official"
        asyncio.create_task(_process_one_event(event))
    return {"message": "OK"}
'''
print(code)

## สรุป

```
LINE OA Flow:

User พิมพ์ข้อความ
    ↓
LINE Platform ส่ง POST ไปที่ webhook URL ของเรา
    ↓
extract_line_event() → normalize JSON → dict {
    event_type, reply_token, user_id, 
    message_text, postback_data
}
    ↓
show_loading_animation() ← ผู้ใช้เห็นจุด 3 จุด
    ↓
AI Processing... (อาจใช้เวลาหลายวินาที)
    ↓
reply_to_line(reply_token, text/flex/quick_replies)
    ↓
ผู้ใช้เห็นคำตอบ
```

**Notebook ถัดไป:** `02_fastapi_setup.ipynb` — ดูว่า FastAPI รับ webhook และ route ได้อย่างไร